# GenX Capacity Expansion Runner

In [ ]:
from ipywidgets import Dropdown, SelectMultiple
from upath import UPath
from src import runner
from ipyfilechooser import FileChooser
import xlwings as xw
from loguru import logger
from tqdm.notebook import trange, tqdm

In [ ]:
genx_wb = FileChooser(default_path=".", default_filename="Kentucky Load Resource Model.xlsb", title="Connect to a GenX spreadsheet: ", filter_pattern="*.xls*", show_hidden=False)
genx_wb

In [ ]:
logger.info(f"Opening {genx_wb.value} in new Excel instance")

folders = []

if xw.apps:
    if UPath(genx_wb.value).parts[-1] in xw.books:
        xw.books[UPath(genx_wb.value).parts[-1]].save()

with xw.App(visible=False) as xw_sandbox:
    inputs_wb = xw.apps[xw_sandbox.pid].books.open(genx_wb.value)
    
    cases_to_run = [c for c in inputs_wb.sheets["Batch Cases"].range("CasesToRun").options(empty=None).value if c is not None]
    
    inputs_wb.screen_updating = False    
    inputs_wb.app.calculate()
    
    for case_name in tqdm(
        cases_to_run,
        desc="Saving CEM cases",
    ): 
        folders.append(runner.update_params_and_save_case(params={"CaseName": case_name}, wb=inputs_wb))
    
    inputs_wb.close()

# Run CEM cases in parallel
results = runner.run_cases_with_streaming_logs(folders, n_lines=8, max_parallel=4)

unsuccessful_cases = [p for p in folders if p.is_dir() and not (p / "results" / "capacities_multi_stage.csv").exists()]
num_successful_cases = len(folders) - len(unsuccessful_cases)

logger.success(f"Successfully solved {num_successful_cases} out of {len(folders)}!")
if unsuccessful_cases:
    logger.warning(f"Failed to solved the following cases: {unsuccessful_cases}")